In [10]:
import pandas as pd
from functools import partial

dataset_root = "/home/xzhao/workspace/GYB_self-ensemble/datasets"
def get_ppl_filename(
        model_name, dataset_name, logits_ensemble_method, 
        is_baseline, repeat_paras, ensemble_method, 
        ensemble_layer, ensemble_alpha, token_mode, 
        multilayer, num_fewshots, num_paraphrases):
    
    if is_baseline:
        dump_file = f"{dataset_root}/{dataset_name}_paraphrase/{model_name}/{dataset_name}paraphrase.ppl.baseline."
    else:
        dump_file = f"{dataset_root}/{dataset_name}_paraphrase/{model_name}/{dataset_name}paraphrase.ppl.logits.{logits_ensemble_method}."
        if ensemble_method == "layer_output_avg":
            dump_file += f"avglayer.layer{ensemble_layer}.alpha{int(ensemble_alpha*100)}.token-{token_mode}."
        elif ensemble_method == "ffn_activation_max":
            dump_file += f"maxffn.layer{ensemble_layer}.alpha{int(ensemble_alpha*100)}.token-{token_mode}."
        if multilayer:
            dump_file += "multilayer."
    if num_fewshots != 5:
        dump_file += f"{num_fewshots}fshots."
    dump_file += f"{num_paraphrases}paras.feather"
    return dump_file

In [11]:
num_fewshots = 0
num_paraphrases = 3

get_ppl_filename_partial = partial(
    get_ppl_filename,
    logits_ensemble_method="avg",
    num_fewshots=num_fewshots,
    num_paraphrases=num_paraphrases,
    multilayer=True,
    repeat_paras=False,
    ensemble_method="layer_output_avg", 
    ensemble_alpha=1, 
    token_mode="last")

In [13]:
from notebooks._utils import calculate_accuracy, get_layers

# model_name = "llama3.2_3b"
for model_name in ["llama3.2_3b", "qwen2.5_3b", "qwen3_4b", "pythia_2.8b", "qwen3_30b"]:
    print(f"\n================ Evaluating model: {model_name} ================ ")
    # for dataset in ["commonsense", "mmlu", "logiqa"]: 
    for dataset in ["mmlu"]: 
        print(f"\n------ Evaluating dataset: {dataset} ------")
        basefn = get_ppl_filename_partial(
            model_name=model_name,
            dataset_name=dataset,
            is_baseline=True,
            ensemble_layer=get_layers(model_name))
        
        ensemblefn = get_ppl_filename_partial(
            model_name=model_name,
            dataset_name=dataset,
            is_baseline=False,
            ensemble_layer=get_layers(model_name))
        
        try:
            basedf = pd.read_feather(basefn)
            calculate_accuracy(basedf, label="Baseline", is_multichoice=True)
        except FileNotFoundError:
            print(f"Baseline file not found: {basefn}")

        try:
            ensembledf = pd.read_feather(ensemblefn)
            calculate_accuracy(ensembledf, label="Ensemble", is_multichoice=True)
        except FileNotFoundError:
            print(f"Ensemble file not found: {ensemblefn}")


================ Evaluating model: llama3.2_3b ================ 

------ Evaluating dataset: mmlu ------
Multichoice Acc: 0.4033 ==> 🏷️ Baseline
Multichoice Acc: 0.4070 ==> 🏷️ Ensemble

================ Evaluating model: qwen2.5_3b ================ 

------ Evaluating dataset: mmlu ------
Multichoice Acc: 0.4390 ==> 🏷️ Baseline
Ensemble file not found: /home/xzhao/workspace/GYB_self-ensemble/datasets/mmlu_paraphrase/qwen2.5_3b/mmluparaphrase.ppl.logits.avg.avglayer.layer27.alpha100.token-last.multilayer.0fshots.3paras.feather

================ Evaluating model: qwen3_4b ================ 

------ Evaluating dataset: mmlu ------
Multichoice Acc: 0.4193 ==> 🏷️ Baseline
Ensemble file not found: /home/xzhao/workspace/GYB_self-ensemble/datasets/mmlu_paraphrase/qwen3_4b/mmluparaphrase.ppl.logits.avg.avglayer.layer27.alpha100.token-last.multilayer.0fshots.3paras.feather

================ Evaluating model: pythia_2.8b ================ 

------ Evaluating dataset: mmlu ------
Multichoice Acc: 0

In [7]:
import os
os.path.exists(basefn)

False

In [14]:
basedf

,uuid,answers,prediction,generation,correctness,paraphrases,ppls,best_choice_idx,choices_label,choices_text,answer_label
0,000d45b3136678e85a763c55a797fe44,two,A,A,None,[For 0 ≤ t ≤ 10 with velocity v(t) = t cos(t) ...,"[22380.01171875, 41092.30859375, 71967.5390625...",none,"[A, B, C, D]","[none, one, two, three]",C
1,000d45b3136678e85a763c55a797fe44,two,A,A,None,[How many times does the particle change its d...,"[33044.51953125, 583530.3125, 2506643.5, 20218...",none,"[A, B, C, D]","[none, one, two, three]",C
2,000d45b3136678e85a763c55a797fe44,two,A,A,None,"[Over t between 0 and 10, how often does the m...","[45081.8984375, 55084.41796875, 97161.9765625,...",none,"[A, B, C, D]","[none, one, two, three]",C
3,002fba5ca35de3b4c7398c42e18b04e2,results in an equilibrium that does not maximi...,D,D,None,[What does the term external cost or benefit r...,"[58.999019622802734, 55.22714614868164, 795.42...",results in an equilibrium that does not maximi...,"[A, B, C, D]",[causes the equilibrium price to be artificial...,D
4,002fba5ca35de3b4c7398c42e18b04e2,results in an equilibrium that does not maximi...,D,D,None,[What is meant by an externality in economic t...,"[50.98200988769531, 47.15764236450195, 1030.26...",results in an equilibrium that does not maximi...,"[A, B, C, D]",[causes the equilibrium price to be artificial...,D
...,...,...,...,...,...,...,...,...,...,...,...
2995,ffcadb4334eae438177df628ec1b67a5,increasing water lost through skin,B,B,None,[How does the body respond when the surroundin...,"[4302.669921875, 28.24742889404297, 86.5704421...",increasing respiration rate,"[A, B, C, D]","[decreasing salt retention, increasing respira...",D
2996,ffcadb4334eae438177df628ec1b67a5,increasing water lost through skin,B,B,None,[What physiological change occurs in humans to...,"[2317.986083984375, 96.51834869384766, 205.095...",increasing respiration rate,"[A, B, C, D]","[decreasing salt retention, increasing respira...",D
2997,ffdafde5e12f3be106c377bd86cf1cc1,ten thousand,D,D,None,[How many hundred-dollar notes are needed to m...,"[109.7259292602539, 1412.18505859375, 143.0312...",one hundred thousand,"[A, B, C, D]","[one thousand, five thousand, ten thousand, on...",C
2998,ffdafde5e12f3be106c377bd86cf1cc1,ten thousand,D,D,None,"[If you only used $100 denominations, how many...","[95.23861694335938, 141.4641571044922, 56.1894...",one hundred thousand,"[A, B, C, D]","[one thousand, five thousand, ten thousand, on...",C
